# Whisper Realtime Computer Audio Demo

Notebook kiểm thử nhận dạng giọng nói realtime bằng Whisper, lấy nguồn âm thanh trực tiếp từ máy tính (loopback/stereo mix).

**Outline:**
1. Cài đặt và nhập các thư viện cần thiết
2. Tạo module thu âm thanh từ máy tính
3. Tạo module xử lý và truyền dữ liệu âm thanh
4. Tích hợp Whisper để nhận dạng giọng nói theo thời gian thực
5. Notebook kiểm thử: Thu và nhận dạng âm thanh realtime

## 1. Cài đặt và nhập các thư viện cần thiết
Cài đặt sounddevice, numpy, whisper, và các phụ thuộc khác.

In [ ]:
# Cài đặt các thư viện cần thiết
%pip install -q sounddevice numpy torch openai-whisper

import sounddevice as sd
import numpy as np
import whisper
import sys
print('sounddevice:', sd.__version__)
print('numpy:', np.__version__)
print('whisper:', whisper.__version__)
print('Python:', sys.version)

## 2. Tạo module thu âm thanh từ máy tính
Sử dụng sounddevice để thu âm thanh từ thiết bị đầu ra (loopback/stereo mix).

In [ ]:
# Module thu âm thanh từ thiết bị đầu ra (loopback/stereo mix)
import sounddevice as sd
import numpy as np

def record_computer_audio(duration=10, samplerate=16000, channels=1, device=None):
    print(f"Recording {duration}s from computer audio (loopback/stereo mix)...")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=channels, dtype='float32', device=device)
    sd.wait()
    return audio.flatten()

# Lấy danh sách thiết bị để chọn đúng thiết bị loopback/stereo mix
print("Available audio devices:")
print(sd.query_devices())

## 3. Xử lý và truyền dữ liệu âm thanh
Chuyển đổi dữ liệu sang định dạng phù hợp cho Whisper, chia đoạn nếu cần.

In [ ]:
# Hàm chuẩn hóa và chia đoạn audio (nếu cần)
def preprocess_audio(audio, target_sr=16000):
    # Nếu audio không phải float32, chuyển đổi
    if audio.dtype != np.float32:
        audio = audio.astype(np.float32)
    # Chuẩn hóa biên độ
    audio = audio / np.max(np.abs(audio))
    return audio

# Hàm chia đoạn (chunk) nếu muốn nhận dạng realtime từng phần
def chunk_audio(audio, chunk_size=16000*5):  # 5s mỗi chunk
    return [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]

## 4. Tích hợp Whisper để nhận dạng giọng nói theo thời gian thực
Sử dụng mô hình Whisper để nhận dạng từng đoạn audio.

In [ ]:
# Hàm nhận dạng từng chunk bằng Whisper
import whisper

def transcribe_chunks(chunks, model_name="small", language="vi", device="cpu"):
    model = whisper.load_model(model_name, device=device)
    results = []
    for i, chunk in enumerate(chunks):
        print(f"Transcribing chunk {i+1}/{len(chunks)}...")
        result = model.transcribe(chunk, language=language, fp16=False, task="transcribe", verbose=False)
        print("Transcript:", result["text"])
        results.append(result["text"])
    return results

## 5. Notebook kiểm thử: Thu và nhận dạng âm thanh realtime
Ghi âm từ máy tính, chia đoạn, nhận dạng và hiển thị kết quả.

In [ ]:
# Thông số
DURATION = 15  # Thời gian ghi âm (giây)
SAMPLERATE = 16000
MODEL_NAME = "small"
LANGUAGE = "vi"

# 1. Ghi âm từ máy tính (chọn đúng device nếu cần)
audio = record_computer_audio(duration=DURATION, samplerate=SAMPLERATE)

# 2. Xử lý và chia đoạn
audio = preprocess_audio(audio, target_sr=SAMPLERATE)
chunks = chunk_audio(audio, chunk_size=SAMPLERATE*5)  # 5s mỗi chunk

# 3. Nhận dạng từng chunk
results = transcribe_chunks(chunks, model_name=MODEL_NAME, language=LANGUAGE)

# 4. Hiển thị kết quả cuối
full_text = " ".join(results)
print("\nFull transcript:")
print(full_text)